# Dia 10 — pytest para Ciência de Dados 🧪
## Testes unitários aplicados a funções de PLN e processamento de texto

**Fase 1 — Fundamentos de Python** · **Data:** 23/10/2026 · **Autor:** Gutemberg Rapôso

Nos dias anteriores eu escrevi muitas funções: limpeza de strings, regex, classes, manipulação de datas.
Mas **como eu sei que elas funcionam** — e continuam funcionando depois que eu mudo alguma coisa?
A resposta profissional é: **testes automatizados**. Hoje aprendo `pytest`, o framework de testes mais usado no ecossistema Python,
aplicado ao meu contexto de Linguística e PLN.

### 🎯 Objetivos de aprendizagem

1. Entender **por que** cientistas de dados precisam testar código (reprodutibilidade, confiança, código de produção).
2. Escrever testes com `assert` simples e ler a saída do pytest (inclusive quando o teste **falha**).
3. Testar exceções com `pytest.raises`.
4. Usar **fixtures** (incluindo `tmp_path`, escopos e `yield` para *teardown*).
5. Usar `@pytest.mark.parametrize` para testar muitos casos (e **valores-limite**) sem repetir código.
6. Testar funções que retornam **DataFrames e arrays** (`pandas.testing`, `numpy.testing`, `pytest.approx`).
7. Organizar fixtures compartilhadas em `conftest.py` e usar **markers** (`slow`, `-k`, `-m`).
8. Medir **cobertura de código** com `pytest-cov` e interpretar o relatório.
9. Rodar testes dentro do notebook com `ipytest`.

**Dados reais usados:** corpus *Gutenberg* e lista de *stopwords* do NLTK, e o dataset *20 Newsgroups* do scikit-learn.

In [1]:
# Core imports and environment check
import os
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import pytest
import sklearn

print(f"Python      : {sys.version.split()[0]}")
print(f"pytest      : {pytest.__version__}")
print(f"nltk        : {nltk.__version__}")
print(f"pandas      : {pd.__version__}")
print(f"numpy       : {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

Python      : 3.11.6
pytest      : 9.1.1
nltk        : 3.10.3
pandas      : 2.2.3
numpy       : 1.26.4
scikit-learn: 1.9.1


### 📚 Os datasets (todos públicos e reais)

| Dataset | Fonte | Uso no notebook |
|---|---|---|
| `gutenberg` | NLTK (18 textos literários em inglês do Projeto Gutenberg) | fixtures com texto real e ranking de frequência |
| `stopwords` | NLTK (lista de palavras funcionais do inglês) | teste de remoção de stopwords |
| `20 Newsgroups` | scikit-learn (mensagens reais de fóruns Usenet) | testes de robustez em textos "sujos" (e-mails, URLs, números) |

Textos literários são relativamente "limpos"; mensagens de fórum têm ruído de verdade. Testar nos dois tipos é uma boa prática:
o código precisa sobreviver aos dados do mundo real.

In [2]:
# Download the NLTK resources (quiet, idempotent) and load the corpora
from nltk.corpus import gutenberg, stopwords
from sklearn.datasets import fetch_20newsgroups

for resource in ["gutenberg", "stopwords"]:
    nltk.download(resource, quiet=True)

NEWS_CATEGORIES = ["sci.med", "rec.sport.baseball"]
news = fetch_20newsgroups(
    subset="train",
    categories=NEWS_CATEGORIES,
    remove=("headers", "footers", "quotes"),
)

print("Gutenberg files :", len(gutenberg.fileids()))
print("Stopwords (en)  :", len(stopwords.words("english")))
print("Newsgroups docs :", len(news.data), "| categories:", news.target_names)

Gutenberg files : 18
Stopwords (en)  : 198
Newsgroups docs : 1191 | categories: ['rec.sport.baseball', 'sci.med']


In [3]:
# A quick look at the raw material we are going to test against
emma_raw = gutenberg.raw("austen-emma.txt")
print(emma_raw[:300])
print("-" * 60)
print(repr(news.data[0][:300]))

[Emma by Jane Austen 1816]

VOLUME I

CHAPTER I


Emma Woodhouse, handsome, clever, and rich, with a comfortable home
and happy disposition, seemed to unite some of the best blessings
of existence; and had lived nearly twenty-one years in the world
with very little to distress or vex her.

She was t
------------------------------------------------------------
'I am looking for a source of American League baseball stats for\nindividual players in the same format as printed in newspapers, ie. I do\nnot want to provide a list of players and get back nice printed reports\nfor $35 a week.\n\nDoes anyone know of such statistics availability and an idea of the\ncost?'


### 🤔 Por que testar em Ciência de Dados?

Em notebooks, é comum "testar" olhando o resultado de uma célula. O problema:

- **Regressões silenciosas:** mudo uma regex para resolver um caso e quebro outros três — sem perceber.
- **Dados mudam:** um novo lote de textos traz acentos, emojis ou URLs que eu nunca vi.
- **Reprodutibilidade:** um pipeline de PLN alimenta modelos; um tokenizador errado contamina **todas** as métricas depois.
- **Mercado:** vagas de Data Scientist pedem cada vez mais *production-ready code* — testes são o mínimo esperado.

**Estratégia deste notebook:** escrever um pequeno módulo `textprep.py` (como se fosse um pacote real),
e arquivos `test_*.py` separados, executados pelo pytest via linha de comando — exatamente como num projeto profissional.
Tudo acontece em uma pasta temporária, para o notebook ser autocontido.

In [4]:
# Lab workspace: a temporary folder that plays the role of a small project
WORKDIR = Path(tempfile.mkdtemp(prefix="pytest_ds_lab_"))


def write_file(name: str, content: str) -> Path:
    """Write ``content`` (dedented) to ``WORKDIR/name`` and report its size."""
    path = WORKDIR / name
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")
    n_lines = len(path.read_text(encoding="utf-8").splitlines())
    print(f"wrote {path.name} ({n_lines} lines)")
    return path


def run_pytest(*args: str, max_chars: int = 5000) -> int:
    """Run pytest in a subprocess inside WORKDIR and print its output.

    A subprocess guarantees that every run imports fresh copies of the
    module and the tests, just like running ``pytest`` in a terminal.
    """
    cmd = [sys.executable, "-m", "pytest", "-p", "no:cacheprovider", *args]
    env = {**os.environ, "PYTHONPATH": str(WORKDIR)}
    result = subprocess.run(cmd, cwd=WORKDIR, capture_output=True, text=True, env=env)
    print(result.stdout[-max_chars:])
    if result.stderr.strip():
        print("STDERR:", result.stderr[-2000:])
    print(f"exit code: {result.returncode}")
    return result.returncode


print("Workspace:", WORKDIR.name)

Workspace: pytest_ds_lab_v6_b40jk


### 🧩 O código a ser testado: módulo `textprep.py`

Um mini-pacote de pré-processamento com funções típicas de PLN:

- `clean_text` — normaliza Unicode, remove URLs/e-mails/números/pontuação, preserva apóstrofos internos (*don't*, *Emma's*).
- `tokenize`, `remove_stopwords` — tokenização simples por espaço e filtro de palavras funcionais.
- `word_frequencies`, `type_token_ratio`, `normalize_counts` — análise de frequência e diversidade lexical (pandas/numpy).
- `build_rank_lookup`, `cefr_band_from_rank`, `cefr_classify`, `lexical_profile` — um **classificador de nível CEFR por faixa de frequência**.

⚠️ **Nota metodológica (honestidade linguística):** as faixas A1–C2 aqui são um **proxy baseado em frequência** em um corpus
(palavras mais frequentes tendem a ser aprendidas antes). **Não** é o *English Vocabulary Profile* oficial nem uma classificação validada.
O objetivo é ter uma função com regras claras e **valores-limite** — perfeita para praticar testes.

In [5]:
# Write the module under test
write_file("textprep.py", r'''
"""Text preprocessing utilities for NLP-oriented data science.

The CEFR functions implement a *frequency-band proxy*: words are assigned to
A1-C2 according to their frequency rank in a reference corpus. This is a
teaching heuristic, not an official CEFR vocabulary list.
"""
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd

_URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
_EMAIL_RE = re.compile(r"\S+@\S+")
_NON_LETTER_RE = re.compile(r"[^A-Za-z'\s]")
_LOOSE_APOSTROPHE_RE = re.compile(r"(?<![A-Za-z])'|'(?![A-Za-z])")
_SPACES_RE = re.compile(r"\s+")

# Upper rank limit (inclusive) for each band; ranks above the last limit -> C2
CEFR_BAND_LIMITS = [(500, "A1"), (1000, "A2"), (2000, "B1"), (4000, "B2"), (8000, "C1")]
CEFR_BANDS = ["A1", "A2", "B1", "B2", "C1", "C2", "UNKNOWN"]


def clean_text(text, lowercase=True):
    """Normalize raw text into letters, internal apostrophes and single spaces.

    Steps: curly quotes -> straight, accents stripped (NFKD), URLs and e-mails
    removed, optional lowercasing, non-letters removed, loose apostrophes
    removed, whitespace collapsed.

    Raises:
        TypeError: if ``text`` is not a string.
    """
    if not isinstance(text, str):
        raise TypeError(f"text must be str, got {type(text).__name__}")
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = _URL_RE.sub(" ", text)
    text = _EMAIL_RE.sub(" ", text)
    if lowercase:
        text = text.lower()
    text = _NON_LETTER_RE.sub(" ", text)
    text = _LOOSE_APOSTROPHE_RE.sub(" ", text)
    return _SPACES_RE.sub(" ", text).strip()


def tokenize(text, min_len=1):
    """Clean ``text`` and split it into tokens of at least ``min_len`` chars."""
    if min_len < 1:
        raise ValueError("min_len must be >= 1")
    return [tok for tok in clean_text(text).split() if len(tok) >= min_len]


def remove_stopwords(tokens, stopword_set):
    """Return tokens that are not in ``stopword_set`` (order preserved)."""
    return [tok for tok in tokens if tok not in stopword_set]


def word_frequencies(tokens, top_n=None):
    """Return a DataFrame with columns token, count, relative_freq.

    Sorted by count (descending) and then alphabetically for ties.
    """
    columns = ["token", "count", "relative_freq"]
    if not tokens:
        return pd.DataFrame({
            "token": pd.Series(dtype="object"),
            "count": pd.Series(dtype="int64"),
            "relative_freq": pd.Series(dtype="float64"),
        })[columns]
    counts = Counter(tokens)
    items = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    if top_n is not None:
        items = items[:top_n]
    df = pd.DataFrame(items, columns=["token", "count"])
    df["count"] = df["count"].astype("int64")
    df["relative_freq"] = df["count"] / len(tokens)
    return df[columns]


def type_token_ratio(tokens):
    """Lexical diversity: number of distinct types / number of tokens."""
    if len(tokens) == 0:
        raise ValueError("type_token_ratio is undefined for an empty token list")
    return len(set(tokens)) / len(tokens)


def normalize_counts(counts):
    """Convert raw counts to frequencies per million words (numpy array)."""
    arr = np.asarray(counts, dtype=float)
    if (arr < 0).any():
        raise ValueError("counts must be non-negative")
    total = arr.sum()
    if total == 0:
        raise ValueError("counts must not sum to zero")
    return arr / total * 1_000_000


def build_rank_lookup(tokens):
    """Map each token to its 1-based frequency rank (ties broken alphabetically)."""
    counts = Counter(tokens)
    ordered = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    return {tok: rank for rank, (tok, _) in enumerate(ordered, start=1)}


def cefr_band_from_rank(rank):
    """Map a frequency rank (1 = most frequent) to a CEFR-like band."""
    if rank < 1:
        raise ValueError("rank must be >= 1")
    for limit, band in CEFR_BAND_LIMITS:
        if rank <= limit:
            return band
    return "C2"


def cefr_classify(word, rank_lookup):
    """Classify a word with the frequency-band proxy; unseen words -> UNKNOWN."""
    rank = rank_lookup.get(clean_text(word))
    return "UNKNOWN" if rank is None else cefr_band_from_rank(rank)


def lexical_profile(tokens, rank_lookup):
    """Count and share of tokens per band, always listing every band in order."""
    bands = pd.Series([cefr_classify(t, rank_lookup) for t in tokens], dtype="object")
    counts = bands.value_counts().reindex(CEFR_BANDS, fill_value=0).astype("int64")
    total = int(counts.sum())
    share = counts / total if total else counts.astype(float)
    return pd.DataFrame({"band": CEFR_BANDS, "count": counts.values, "share": share.values})
''')

wrote textprep.py (129 lines)


PosixPath('/tmp/pytest_ds_lab_v6_b40jk/textprep.py')

In [6]:
# Import the module in the notebook too, and try it on real text
sys.path.insert(0, str(WORKDIR))
import textprep  # noqa: E402

emma_tokens = textprep.tokenize(emma_raw[:2000])
print(emma_tokens[:20])
print("TTR (first 2,000 chars of Emma):", round(textprep.type_token_ratio(emma_tokens), 3))
textprep.word_frequencies(emma_tokens, top_n=8)

['emma', 'by', 'jane', 'austen', 'volume', 'i', 'chapter', 'i', 'emma', 'woodhouse', 'handsome', 'clever', 'and', 'rich', 'with', 'a', 'comfortable', 'home', 'and', 'happy']
TTR (first 2,000 chars of Emma): 0.564


,token,count,relative_freq
0,of,20,0.058140
1,the,16,0.046512
2,her,13,0.037791
3,and,11,0.031977
4,a,9,0.026163
5,had,9,0.026163
6,to,7,0.020349
7,in,6,0.017442


### 1️⃣ O básico do pytest: funções `test_*` e `assert`

Regras de descoberta (*test discovery*) padrão do pytest:

- arquivos chamados `test_*.py` ou `*_test.py`;
- funções cujo nome começa com `test_`;
- dentro delas, um simples `assert` — sem `self.assertEqual` como no `unittest`.

Cada teste deve verificar **um comportamento** e ter um nome que funcione como documentação:
`test_clean_text_keeps_internal_apostrophes` diz exatamente o que se espera.

In [7]:
write_file("test_basics.py", r'''
"""Basic pytest examples for textprep: plain assert statements."""
from textprep import clean_text, tokenize, type_token_ratio


def test_clean_text_lowercases():
    assert clean_text("Emma Woodhouse") == "emma woodhouse"


def test_clean_text_can_preserve_case():
    assert clean_text("Emma Woodhouse", lowercase=False) == "Emma Woodhouse"


def test_clean_text_removes_punctuation_and_digits():
    assert clean_text("Chapter 1. Emma, handsome!") == "chapter emma handsome"


def test_clean_text_keeps_internal_apostrophes():
    assert clean_text("Don't 'quote' me") == "don't quote me"


def test_tokenize_returns_expected_tokens():
    tokens = tokenize("Emma Woodhouse, handsome, clever, and rich")
    assert tokens == ["emma", "woodhouse", "handsome", "clever", "and", "rich"]


def test_tokenize_min_len_filters_short_tokens():
    assert tokenize("I am a clever girl", min_len=3) == ["clever", "girl"]


def test_type_token_ratio_simple_case():
    assert type_token_ratio(["the", "cat", "the", "dog"]) == 0.75
''')

run_pytest("test_basics.py", "-v")

wrote test_basics.py (31 lines)


============================= test session starts ==============================
platform linux -- Python 3.11.6, pytest-9.1.1, pluggy-1.7.0 -- /opt/computersetup/.pyenv/versions/3.11.6/bin/python
rootdir: /tmp/pytest_ds_lab_v6_b40jk
plugins: cov-7.1.0, platformdirs-4.13.5, anyio-4.15.1
collecting ... collected 7 items

test_basics.py::test_clean_text_lowercases PASSED                        [ 14%]
test_basics.py::test_clean_text_can_preserve_case PASSED                 [ 28%]
test_basics.py::test_clean_text_removes_punctuation_and_digits PASSED    [ 42%]
test_basics.py::test_clean_text_keeps_internal_apostrophes PASSED        [ 57%]
test_basics.py::test_tokenize_returns_expected_tokens PASSED             [ 71%]
test_basics.py::test_tokenize_min_len_filters_short_tokens PASSED        [ 85%]
test_basics.py::test_type_token_ratio_simple_case PASSED                 [100%]

============================== 7 passed in 0.26s ===============================

exit code: 0


0

**Lendo a saída:** `-v` (*verbose*) lista cada teste com `PASSED`/`FAILED`. O *exit code* `0` significa "tudo passou" —
é esse código que ferramentas de CI (GitHub Actions, por exemplo) usam para aprovar ou bloquear um *merge*.

### Quando um teste falha
A grande vantagem do pytest é a **introspecção de asserts**: ele mostra os valores dos dois lados da comparação.
Vou escrever, de propósito, um teste com a expectativa errada (TTR de `"the the the cat"` é 2/4 = 0.5, não 0.75).

In [8]:
# Deliberately failing test, to see pytest's failure report (exit code 1 is expected)
write_file("test_failure_demo.py", r'''
from textprep import tokenize, type_token_ratio


def test_ttr_with_wrong_expectation():
    tokens = tokenize("the the the cat")
    assert type_token_ratio(tokens) == 0.75  # wrong on purpose: real value is 0.5
''')

exit_code = run_pytest("test_failure_demo.py", "-q", "--tb=short")
assert exit_code == 1, "the demo test was supposed to fail"

# Remove the demo file so it does not pollute the real suite
(WORKDIR / "test_failure_demo.py").unlink()

wrote test_failure_demo.py (6 lines)


F                                                                        [100%]
=================================== FAILURES ===================================
_______________________ test_ttr_with_wrong_expectation ________________________
test_failure_demo.py:6: in test_ttr_with_wrong_expectation
    assert type_token_ratio(tokens) == 0.75  # wrong on purpose: real value is 0.5
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
E   AssertionError: assert 0.5 == 0.75
E    +  where 0.5 = type_token_ratio(['the', 'the', 'the', 'cat'])
=========================== short test summary info ============================
FAILED test_failure_demo.py::test_ttr_with_wrong_expectation - AssertionError: assert 0.5 == 0.75
1 failed in 0.30s

exit code: 1


Repare na linha `assert 0.5 == 0.75`: o pytest reescreve o `assert` e mostra o valor real calculado.
Isso economiza muitos `print()` de depuração. Opções úteis de *traceback*: `--tb=short`, `--tb=line`, `--tb=no`.

### 2️⃣ Testando exceções com `pytest.raises`

Uma função bem projetada **falha de forma explícita** com entradas inválidas (em vez de devolver lixo silenciosamente).
Em PLN isso é crucial: um `NaN` vindo de uma coluna do pandas (`float`) não deveria ser "limpo" como se fosse texto.
O argumento `match=` verifica a mensagem de erro com uma regex.

In [9]:
write_file("test_exceptions.py", r'''
"""Invalid inputs must raise clear exceptions."""
import math

import pytest

from textprep import (
    cefr_band_from_rank,
    clean_text,
    normalize_counts,
    tokenize,
    type_token_ratio,
)


def test_clean_text_rejects_none():
    with pytest.raises(TypeError, match="must be str"):
        clean_text(None)


def test_clean_text_rejects_nan_from_pandas():
    # Missing values in a pandas text column arrive as float('nan')
    with pytest.raises(TypeError, match="float"):
        clean_text(math.nan)


def test_tokenize_rejects_non_positive_min_len():
    with pytest.raises(ValueError, match="min_len"):
        tokenize("some text", min_len=0)


def test_ttr_undefined_for_empty_list():
    with pytest.raises(ValueError, match="empty"):
        type_token_ratio([])


def test_normalize_counts_rejects_negative_values():
    with pytest.raises(ValueError, match="non-negative"):
        normalize_counts([3, -1, 2])


def test_normalize_counts_rejects_all_zero():
    with pytest.raises(ValueError, match="zero"):
        normalize_counts([0, 0, 0])


def test_rank_must_be_positive():
    with pytest.raises(ValueError):
        cefr_band_from_rank(0)
''')

run_pytest("test_exceptions.py", "-v")

wrote test_exceptions.py (48 lines)


============================= test session starts ==============================
platform linux -- Python 3.11.6, pytest-9.1.1, pluggy-1.7.0 -- /opt/computersetup/.pyenv/versions/3.11.6/bin/python
rootdir: /tmp/pytest_ds_lab_v6_b40jk
plugins: cov-7.1.0, platformdirs-4.13.5, anyio-4.15.1
collecting ... collected 7 items

test_exceptions.py::test_clean_text_rejects_none PASSED                  [ 14%]
test_exceptions.py::test_clean_text_rejects_nan_from_pandas PASSED       [ 28%]
test_exceptions.py::test_tokenize_rejects_non_positive_min_len PASSED    [ 42%]
test_exceptions.py::test_ttr_undefined_for_empty_list PASSED             [ 57%]
test_exceptions.py::test_normalize_counts_rejects_negative_values PASSED [ 71%]
test_exceptions.py::test_normalize_counts_rejects_all_zero PASSED        [ 85%]
test_exceptions.py::test_rank_must_be_positive PASSED                    [100%]

============================== 7 passed in 0.26s ===============================

exit code: 0


0

### 3️⃣ Fixtures: preparando dados reais para os testes

Uma **fixture** é uma função decorada com `@pytest.fixture` que fornece dados (ou recursos) aos testes.
O teste simplesmente **declara o nome da fixture como parâmetro** — o pytest faz a injeção de dependência.

Vantagens:
- evita repetir o mesmo *setup* em cada teste;
- fixtures podem depender de outras fixtures (`emma_tokens` depende de `emma_opening`);
- a fixture embutida `tmp_path` fornece uma pasta temporária única para testar leitura/escrita de arquivos.

Aqui as fixtures carregam o início real de *Emma*, de Jane Austen, do corpus Gutenberg do NLTK.

In [10]:
write_file("test_fixtures.py", r'''
"""Fixtures that provide real text from the NLTK Gutenberg corpus."""
import pytest
from nltk.corpus import gutenberg, stopwords

from textprep import remove_stopwords, tokenize, word_frequencies


@pytest.fixture
def emma_opening():
    """First 2,000 characters of Jane Austen's Emma (NLTK Gutenberg corpus)."""
    return gutenberg.raw("austen-emma.txt")[:2000]


@pytest.fixture
def emma_tokens(emma_opening):
    """Tokens derived from the ``emma_opening`` fixture (fixture chaining)."""
    return tokenize(emma_opening)


def test_tokens_are_lowercase(emma_tokens):
    assert all(tok == tok.lower() for tok in emma_tokens)


def test_tokens_have_no_digits_or_punctuation(emma_tokens):
    assert all(tok.replace("'", "").isalpha() for tok in emma_tokens)


def test_protagonist_is_mentioned(emma_tokens):
    assert "emma" in emma_tokens
    assert "woodhouse" in emma_tokens


def test_frequency_counts_sum_to_token_count(emma_tokens):
    freq = word_frequencies(emma_tokens)
    assert freq["count"].sum() == len(emma_tokens)


def test_stopword_removal(emma_tokens):
    english_sw = set(stopwords.words("english"))
    content = remove_stopwords(emma_tokens, english_sw)
    assert 0 < len(content) < len(emma_tokens)
    assert not set(content) & english_sw


def test_frequencies_roundtrip_through_csv(emma_tokens, tmp_path):
    """Built-in ``tmp_path`` fixture: a fresh temporary directory per test."""
    import pandas as pd

    freq = word_frequencies(emma_tokens, top_n=20)
    out = tmp_path / "emma_top20.csv"
    freq.to_csv(out, index=False)
    pd.testing.assert_frame_equal(pd.read_csv(out), freq)
''')

run_pytest("test_fixtures.py", "-v")

wrote test_fixtures.py (52 lines)


============================= test session starts ==============================
platform linux -- Python 3.11.6, pytest-9.1.1, pluggy-1.7.0 -- /opt/computersetup/.pyenv/versions/3.11.6/bin/python
rootdir: /tmp/pytest_ds_lab_v6_b40jk
plugins: cov-7.1.0, platformdirs-4.13.5, anyio-4.15.1
collecting ... collected 6 items

test_fixtures.py::test_tokens_are_lowercase PASSED                       [ 16%]
test_fixtures.py::test_tokens_have_no_digits_or_punctuation PASSED       [ 33%]
test_fixtures.py::test_protagonist_is_mentioned PASSED                   [ 50%]
test_fixtures.py::test_frequency_counts_sum_to_token_count PASSED        [ 66%]
test_fixtures.py::test_stopword_removal PASSED                           [ 83%]
test_fixtures.py::test_frequencies_roundtrip_through_csv PASSED          [100%]

============================== 6 passed in 0.79s ===============================

exit code: 0


0

### Escopo de fixtures e *teardown* com `yield`

Por padrão, uma fixture é recriada **para cada teste** (`scope="function"`). Para recursos caros
(carregar um corpus inteiro, abrir conexão com banco, carregar um modelo), usamos escopos maiores:

| scope | criada uma vez por... |
|---|---|
| `function` (padrão) | teste |
| `class` | classe de testes |
| `module` | arquivo `.py` |
| `session` | execução inteira do pytest |

Com `yield` em vez de `return`, o código **depois** do `yield` roda como *teardown* (limpeza).
A opção `-s` desativa a captura de saída, para vermos os `print()` e a ordem de execução.

In [11]:
write_file("test_scope.py", r'''
"""Demonstrates fixture scopes and yield-based teardown."""
import pytest
from nltk.corpus import gutenberg

from textprep import tokenize


@pytest.fixture(scope="module")
def persuasion_tokens():
    print("\n[setup] tokenizing Austen's Persuasion (once per module)")
    tokens = tokenize(gutenberg.raw("austen-persuasion.txt"))
    yield tokens
    print("\n[teardown] releasing Persuasion tokens")


@pytest.fixture
def fresh_counter():
    print("\n[setup] new counter (once per test)")
    yield {"calls": 0}
    print("[teardown] counter discarded")


def test_corpus_is_large(persuasion_tokens, fresh_counter):
    fresh_counter["calls"] += 1
    assert len(persuasion_tokens) > 80_000
    assert fresh_counter["calls"] == 1


def test_anne_is_frequent(persuasion_tokens, fresh_counter):
    fresh_counter["calls"] += 1
    assert persuasion_tokens.count("anne") > 400
    assert fresh_counter["calls"] == 1  # proves the counter was recreated
''')

run_pytest("test_scope.py", "-v", "-s")

wrote test_scope.py (32 lines)


============================= test session starts ==============================
platform linux -- Python 3.11.6, pytest-9.1.1, pluggy-1.7.0 -- /opt/computersetup/.pyenv/versions/3.11.6/bin/python
rootdir: /tmp/pytest_ds_lab_v6_b40jk
plugins: cov-7.1.0, platformdirs-4.13.5, anyio-4.15.1
collecting ... collected 2 items

test_scope.py::test_corpus_is_large 
[setup] tokenizing Austen's Persuasion (once per module)

[setup] new counter (once per test)
PASSED[teardown] counter discarded

test_scope.py::test_anne_is_frequent 
[setup] new counter (once per test)
PASSED[teardown] counter discarded

[teardown] releasing Persuasion tokens


============================== 2 passed in 0.85s ===============================

exit code: 0


0

Observe na saída: `[setup] tokenizing...` aparece **uma vez** (escopo de módulo), enquanto o contador é criado e
descartado **a cada teste**. Isso mantém os testes **isolados** (um não contamina o outro) e ainda assim rápidos.

### 4️⃣ `@pytest.mark.parametrize`: muitos casos, um único teste

Em limpeza de texto, há dezenas de casos-limite: acentos, apóstrofos tipográficos (’), URLs, e-mails, números, espaços.
Em vez de escrever 10 funções quase iguais, uso `parametrize` — cada tupla vira um **teste independente** no relatório.
O argumento `ids=` dá nomes legíveis a cada caso.

In [12]:
write_file("test_parametrize.py", r'''
"""Parametrized tests: edge cases for cleaning and CEFR band boundaries."""
import pytest

from textprep import cefr_band_from_rank, clean_text, tokenize

CLEAN_CASES = [
    ("Hello, World!", "hello world"),
    ("Visit https://www.gutenberg.org now", "visit now"),
    ("Mail me at someone@example.com", "mail me at"),
    ("Caf\u00e9 na\u00efve r\u00e9sum\u00e9", "cafe naive resume"),
    ("It\u2019s Emma\u2019s book", "it's emma's book"),
    ("   multiple    spaces\n\tand tabs ", "multiple spaces and tabs"),
    ("1816", ""),
    ("", ""),
]
CLEAN_IDS = ["punctuation", "url", "email", "accents", "curly-apostrophe",
             "whitespace", "only-digits", "empty"]


@pytest.mark.parametrize("raw, expected", CLEAN_CASES, ids=CLEAN_IDS)
def test_clean_text_cases(raw, expected):
    assert clean_text(raw) == expected


# Boundary-value analysis: test exactly at, below and above each limit
@pytest.mark.parametrize("rank, band", [
    (1, "A1"), (500, "A1"),
    (501, "A2"), (1000, "A2"),
    (1001, "B1"), (2000, "B1"),
    (2001, "B2"), (4000, "B2"),
    (4001, "C1"), (8000, "C1"),
    (8001, "C2"), (50_000, "C2"),
])
def test_cefr_band_boundaries(rank, band):
    assert cefr_band_from_rank(rank) == band


# Stacked parametrize -> cartesian product (3 texts x 3 min_len = 9 tests)
@pytest.mark.parametrize("text", ["Emma Woodhouse, handsome", "I am a cat", "It's a truth"])
@pytest.mark.parametrize("min_len", [1, 2, 4])
def test_tokenize_respects_min_len(text, min_len):
    assert all(len(tok) >= min_len for tok in tokenize(text, min_len=min_len))
''')

run_pytest("test_parametrize.py", "-v")

wrote test_parametrize.py (42 lines)


============================= test session starts ==============================
platform linux -- Python 3.11.6, pytest-9.1.1, pluggy-1.7.0 -- /opt/computersetup/.pyenv/versions/3.11.6/bin/python
rootdir: /tmp/pytest_ds_lab_v6_b40jk
plugins: cov-7.1.0, platformdirs-4.13.5, anyio-4.15.1
collecting ... collected 29 items

test_parametrize.py::test_clean_text_cases[punctuation] PASSED           [  3%]
test_parametrize.py::test_clean_text_cases[url] PASSED                   [  6%]
test_parametrize.py::test_clean_text_cases[email] PASSED                 [ 10%]
test_parametrize.py::test_clean_text_cases[accents] PASSED               [ 13%]
test_parametrize.py::test_clean_text_cases[curly-apostrophe] PASSED      [ 17%]
test_parametrize.py::test_clean_text_cases[whitespace] PASSED            [ 20%]
test_parametrize.py::test_clean_text_cases[only-digits] PASSED           [ 24%]
test_parametrize.py::test_clean_text_cases[empty] PASSED                 [ 27%]
test_parametrize.py::test_cefr_band_b

0

**Insight linguístico:** os casos `curly-apostrophe` e `accents` vêm de problemas reais — textos copiados de editores
como Word trazem `’` em vez de `'`, e corpora multilíngues têm diacríticos. Sem o teste, `It’s` viraria `its` (outra palavra!).

A técnica de **análise de valores-limite** (testar 500 e 501, 1000 e 1001...) é clássica em engenharia de software:
a maioria dos bugs mora nas fronteiras (`<` vs `<=`).

### 5️⃣ Testando funções de pandas e NumPy

Comparar DataFrames com `==` não funciona bem (retorna outro DataFrame, e floats têm erro de arredondamento). Ferramentas certas:

- `pd.testing.assert_frame_equal` / `assert_series_equal` — compara valores, **dtypes**, índices e nomes de colunas;
- `np.testing.assert_allclose` — compara arrays com tolerância numérica;
- `pytest.approx` — compara floats (ou listas de floats) com tolerância.

In [13]:
write_file("test_dataframes.py", r'''
"""Testing functions that return pandas DataFrames and numpy arrays."""
import numpy as np
import pandas as pd
import pytest

from textprep import CEFR_BANDS, lexical_profile, normalize_counts, word_frequencies


@pytest.fixture
def toy_tokens():
    return ["b", "a", "b", "c", "b", "a"]


def test_word_frequencies_exact_frame(toy_tokens):
    expected = pd.DataFrame({
        "token": ["b", "a", "c"],
        "count": np.array([3, 2, 1], dtype="int64"),
        "relative_freq": [3 / 6, 2 / 6, 1 / 6],
    })
    pd.testing.assert_frame_equal(word_frequencies(toy_tokens), expected)


def test_relative_frequencies_sum_to_one(toy_tokens):
    assert word_frequencies(toy_tokens)["relative_freq"].sum() == pytest.approx(1.0)


def test_top_n_limits_rows(toy_tokens):
    assert list(word_frequencies(toy_tokens, top_n=2)["token"]) == ["b", "a"]


def test_empty_input_returns_empty_frame_with_schema():
    df = word_frequencies([])
    assert df.empty
    assert list(df.columns) == ["token", "count", "relative_freq"]
    assert df["count"].dtype == "int64"


def test_normalize_counts_per_million():
    np.testing.assert_allclose(normalize_counts([1, 3]), [250_000.0, 750_000.0])


def test_normalize_counts_preserves_total():
    result = normalize_counts(np.array([5, 10, 85]))
    assert result.sum() == pytest.approx(1_000_000)
    assert isinstance(result, np.ndarray)


def test_lexical_profile_lists_all_bands_in_order():
    lookup = {"the": 1, "house": 600, "melancholy": 2500}
    profile = lexical_profile(["the", "the", "house", "melancholy", "zyzzyva"], lookup)
    assert list(profile["band"]) == CEFR_BANDS
    counts = dict(zip(profile["band"], profile["count"]))
    assert counts == {"A1": 2, "A2": 1, "B1": 0, "B2": 1, "C1": 0, "C2": 0, "UNKNOWN": 1}
    assert profile["share"].sum() == pytest.approx(1.0)
''')

run_pytest("test_dataframes.py", "-v")

wrote test_dataframes.py (54 lines)


============================= test session starts ==============================
platform linux -- Python 3.11.6, pytest-9.1.1, pluggy-1.7.0 -- /opt/computersetup/.pyenv/versions/3.11.6/bin/python
rootdir: /tmp/pytest_ds_lab_v6_b40jk
plugins: cov-7.1.0, platformdirs-4.13.5, anyio-4.15.1
collecting ... collected 7 items

test_dataframes.py::test_word_frequencies_exact_frame PASSED             [ 14%]
test_dataframes.py::test_relative_frequencies_sum_to_one PASSED          [ 28%]
test_dataframes.py::test_top_n_limits_rows PASSED                        [ 42%]
test_dataframes.py::test_empty_input_returns_empty_frame_with_schema PASSED [ 57%]
test_dataframes.py::test_normalize_counts_per_million PASSED             [ 71%]
test_dataframes.py::test_normalize_counts_preserves_total PASSED         [ 85%]
test_dataframes.py::test_lexical_profile_lists_all_bands_in_order PASSED [100%]

============================== 7 passed in 0.31s ===============================

exit code: 0


0

Note o teste `test_empty_input_returns_empty_frame_with_schema`: ele garante o **contrato** da função (as colunas e dtypes)
mesmo sem dados. Isso evita erros como `KeyError: 'count'` lá na frente do pipeline quando um documento vem vazio.

### 6️⃣ `conftest.py`: fixtures compartilhadas e configuração do projeto

Quando várias suítes precisam dos mesmos dados (stopwords, ranking de frequência, um corpus), colocamos as fixtures em
`conftest.py`. O pytest descobre esse arquivo **automaticamente** — não é preciso importá-lo.

Padrões comuns em projetos de DS:
- fixtures `scope="session"` para recursos caros (download/carregamento do corpus uma única vez);
- hooks como `pytest_configure` (aqui, para registrar o marker `slow`);
- um `pytest.ini` (ou `[tool.pytest.ini_options]` no `pyproject.toml`) com opções padrão.

In [14]:
write_file("pytest.ini", r'''
[pytest]
testpaths = .
addopts = -ra
''')

write_file("conftest.py", r'''
"""Shared fixtures for the textprep test suite (auto-discovered by pytest)."""
import pytest
from nltk.corpus import gutenberg, stopwords
from sklearn.datasets import fetch_20newsgroups

from textprep import build_rank_lookup, tokenize


def pytest_configure(config):
    """Register custom markers so pytest does not warn about them."""
    config.addinivalue_line("markers", "slow: tests that iterate over a full corpus")


@pytest.fixture(scope="session")
def english_stopwords():
    """NLTK English stopword list as an immutable set."""
    return frozenset(stopwords.words("english"))


@pytest.fixture(scope="session")
def gutenberg_rank_lookup():
    """Frequency ranks computed over the whole NLTK Gutenberg corpus."""
    return build_rank_lookup(tokenize(gutenberg.raw()))


@pytest.fixture(scope="session")
def newsgroups_docs():
    """Real, noisy forum posts from two 20 Newsgroups categories."""
    data = fetch_20newsgroups(
        subset="train",
        categories=["sci.med", "rec.sport.baseball"],
        remove=("headers", "footers", "quotes"),
    )
    return data.data
''')

wrote pytest.ini (3 lines)
wrote conftest.py (34 lines)


PosixPath('/tmp/pytest_ds_lab_v6_b40jk/conftest.py')

In [15]:
write_file("test_corpus.py", r'''
"""Tests that rely on shared real-corpus fixtures from conftest.py."""
import pytest

from textprep import (
    CEFR_BAND_LIMITS,
    cefr_classify,
    clean_text,
    lexical_profile,
    remove_stopwords,
    tokenize,
    type_token_ratio,
)


def test_most_frequent_gutenberg_word_is_the(gutenberg_rank_lookup):
    assert gutenberg_rank_lookup["the"] == 1


def test_ranks_are_unique_and_contiguous(gutenberg_rank_lookup):
    ranks = sorted(gutenberg_rank_lookup.values())
    assert ranks == list(range(1, len(ranks) + 1))


@pytest.mark.parametrize("word, band", [
    ("the", "A1"),
    ("House", "A1"),          # case-insensitive lookup
    ("happiness", "A2"),
    ("melancholy", "B2"),
    ("benevolent", "C1"),
    ("serendipity", "UNKNOWN"),
])
def test_cefr_classify_gutenberg(word, band, gutenberg_rank_lookup):
    assert cefr_classify(word, gutenberg_rank_lookup) == band


def test_stopwords_are_mostly_a1(gutenberg_rank_lookup, english_stopwords):
    """Linguistic sanity check: function words should be high-frequency.

    NLTK's list also contains contractions ("that'll", "needn't") and
    fragments ("ain", "ma") that are rare in Gutenberg, so we check the
    *median* rank, not every word.
    """
    ranks = sorted(gutenberg_rank_lookup[w] for w in english_stopwords
                   if w in gutenberg_rank_lookup)
    median_rank = ranks[len(ranks) // 2]
    assert cefr_classify("the", gutenberg_rank_lookup) == "A1"
    assert median_rank <= 500  # i.e. the typical stopword falls in band A1


def test_band_limits_are_strictly_increasing():
    limits = [limit for limit, _ in CEFR_BAND_LIMITS]
    assert limits == sorted(limits) and len(set(limits)) == len(limits)


def test_newsgroup_profile_shares_sum_to_one(newsgroups_docs, gutenberg_rank_lookup):
    tokens = tokenize(" ".join(newsgroups_docs[:50]))
    profile = lexical_profile(tokens, gutenberg_rank_lookup)
    assert profile["share"].sum() == pytest.approx(1.0)


@pytest.mark.slow
def test_clean_text_is_idempotent_on_all_newsgroups(newsgroups_docs):
    """Property: cleaning already-clean text must not change it."""
    for doc in newsgroups_docs:
        once = clean_text(doc)
        assert clean_text(once) == once


@pytest.mark.slow
def test_tokens_never_contain_whitespace_or_digits(newsgroups_docs, english_stopwords):
    for doc in newsgroups_docs:
        tokens = remove_stopwords(tokenize(doc), english_stopwords)
        assert all(" " not in t and not any(c.isdigit() for c in t) for t in tokens)
        if tokens:
            assert 0 < type_token_ratio(tokens) <= 1
''')

run_pytest("test_corpus.py", "-v")

wrote test_corpus.py (75 lines)


============================= test session starts ==============================
platform linux -- Python 3.11.6, pytest-9.1.1, pluggy-1.7.0 -- /opt/computersetup/.pyenv/versions/3.11.6/bin/python
rootdir: /tmp/pytest_ds_lab_v6_b40jk
configfile: pytest.ini
plugins: cov-7.1.0, platformdirs-4.13.5, anyio-4.15.1
collecting ... collected 13 items

test_corpus.py::test_most_frequent_gutenberg_word_is_the PASSED          [  7%]
test_corpus.py::test_ranks_are_unique_and_contiguous PASSED              [ 15%]
test_corpus.py::test_cefr_classify_gutenberg[the-A1] PASSED              [ 23%]
test_corpus.py::test_cefr_classify_gutenberg[House-A1] PASSED            [ 30%]
test_corpus.py::test_cefr_classify_gutenberg[happiness-A2] PASSED        [ 38%]
test_corpus.py::test_cefr_classify_gutenberg[melancholy-B2] PASSED       [ 46%]
test_corpus.py::test_cefr_classify_gutenberg[benevolent-C1] PASSED       [ 53%]
test_corpus.py::test_cefr_classify_gutenberg[serendipity-UNKNOWN] PASSED [ 61%]
test_corpus.py

0

Os testes `slow` verificam **propriedades** (invariantes) em **todos** os ~1.200 posts reais do 20 Newsgroups:
- **idempotência**: `clean_text(clean_text(x)) == clean_text(x)`;
- tokens nunca contêm espaços ou dígitos; TTR sempre em (0, 1].

**Lição dos dados reais:** na primeira versão, `test_stopwords_are_mostly_a1` exigia que >70% das stopwords fossem A1 — e
**falhou** (116 de 178 = 65%). Investigando, a lista do NLTK inclui contrações (`that'll`, `needn't`, `she'd`) e fragmentos
(`ain`, `ma`, `y`) que são raros na prosa literária do Gutenberg. O teste revelou uma suposição linguística ingênua; a versão final verifica a **mediana** do ranking.

Esse tipo de teste encontra bugs que exemplos escritos à mão não encontram (é a ideia por trás de bibliotecas
de *property-based testing* como a `hypothesis`).

### 7️⃣ Selecionando testes: markers (`-m`) e palavras-chave (`-k`)

Em projetos grandes, rodamos subconjuntos:
- `-m "not slow"` → pula os testes marcados como lentos (ótimo durante o desenvolvimento);
- `-k cefr` → roda apenas testes cujo nome contém "cefr";
- `-x` → para no primeiro erro; `--lf` → roda só os que falharam da última vez (precisa do cache).

In [16]:
print("=== Fast feedback loop: skip slow tests ===")
run_pytest("-m", "not slow", "-q")

print("\n=== Only CEFR-related tests ===")
run_pytest("-k", "cefr", "-q")

=== Fast feedback loop: skip slow tests ===


.....................................................................    [100%]
69 passed, 2 deselected in 2.69s

exit code: 0

=== Only CEFR-related tests ===


..................                                                       [100%]
18 passed, 53 deselected in 1.74s

exit code: 0


0

A saída de `-m "not slow"` mostra `2 deselected`: os dois testes de corpus completo foram excluídos.
Com `-k cefr`, o pytest selecionou apenas os testes de faixas CEFR (de vários arquivos diferentes).

### 8️⃣ Cobertura de código com `pytest-cov`

**Cobertura** = porcentagem das linhas (e ramos) do código que foram **executadas** pelos testes.

- `--cov=textprep` → mede o módulo `textprep`;
- `--cov-report=term-missing` → mostra os **números das linhas não cobertas**;
- `--cov-branch` → mede também ramos de `if/else` (ex.: o caso `top_n=None` *e* o caso `top_n=5`).

Primeiro, uma suíte **incompleta** (apenas os testes básicos e parametrizados):

In [17]:
run_pytest("test_basics.py", "test_parametrize.py", "-q",
           "--cov=textprep", "--cov-branch", "--cov-report=term-missing")

....................................                                     [100%]
================================ tests coverage ================================
_______________ coverage: platform linux, python 3.11.6-final-0 ________________

Name          Stmts   Miss Branch BrPart  Cover   Missing
---------------------------------------------------------
textprep.py      74     33     22      4    53%   36, 51, 57, 65-79, 85, 91-97, 102-104, 110, 119-120, 125-129
---------------------------------------------------------
TOTAL            74     33     22      4    53%
36 passed in 0.06s

exit code: 0


0

A coluna **Missing** aponta exatamente o que nunca foi exercitado: os `raise` de validação, `word_frequencies`,
`normalize_counts`, o classificador CEFR... Ou seja, cobertura é um **mapa do que falta testar**.

Agora, a suíte **completa** — e com `--cov-fail-under=90`, que faz o pytest **falhar** se a cobertura cair abaixo de 90%
(é assim que equipes impedem que código novo entre sem testes no CI).

In [18]:
coverage_exit = run_pytest("-q", "--cov=textprep", "--cov-branch",
                           "--cov-report=term-missing", "--cov-fail-under=90")
print("Coverage gate passed!" if coverage_exit == 0 else "Coverage gate FAILED")

.......................................................................  [100%]
================================ tests coverage ================================
_______________ coverage: platform linux, python 3.11.6-final-0 ________________

Name          Stmts   Miss Branch BrPart  Cover   Missing
---------------------------------------------------------
textprep.py      74      0     22      0   100%
---------------------------------------------------------
TOTAL            74      0     22      0   100%
Required test coverage of 90% reached. Total coverage: 100.00%
71 passed in 3.49s

exit code: 0
Coverage gate passed!


⚠️ **Cuidado com a métrica:** 100% de cobertura **não** significa código correto — significa apenas que cada linha rodou
pelo menos uma vez. Um teste sem `assert` aumenta a cobertura sem verificar nada. Cobertura é uma ferramenta para
**encontrar lacunas**, não um objetivo em si (Lei de Goodhart: "quando uma medida vira meta, deixa de ser boa medida").

Também é possível gerar relatórios HTML (`--cov-report=html`) ou XML (`--cov-report=xml`, usado por serviços como Codecov).

### 9️⃣ Testes dentro do notebook com `ipytest`

Para experimentar rapidamente, a biblioteca `ipytest` roda o pytest **na própria célula** do Jupyter.
É ótima para estudo e prototipagem; em projetos reais, os testes devem viver em arquivos `tests/`.

In [19]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

In [20]:
%%ipytest -q --color=no

from textprep import clean_text as _clean, cefr_band_from_rank as _band


@pytest.fixture
def alice_opening():
    """Opening of Lewis Carroll's Alice in Wonderland (NLTK Gutenberg)."""
    return gutenberg.raw("carroll-alice.txt")[:1500]


def test_alice_cleaning_removes_brackets(alice_opening):
    assert "[" not in _clean(alice_opening)


def test_alice_is_named(alice_opening):
    assert "alice" in _clean(alice_opening).split()


@pytest.mark.parametrize("rank, band", [(10, "A1"), (750, "A2"), (9999, "C2")])
def test_band_inline(rank, band):
    assert _band(rank) == band

.

.

.

.

.

                                                                                        [100%]

### 🔎 Bônus: o "lado linguístico" do resultado

Com o código testado, posso confiar mais nas análises. Exemplo: o **perfil lexical por faixa de frequência**
de um texto literário (*Emma*) versus posts de fórum de medicina e beisebol. (Lembrete: faixas = proxy de frequência,
não CEFR oficial; e o ranking vem de literatura do séc. XIX, o que penaliza vocabulário técnico moderno.)

In [21]:
# Lexical profile comparison using the (now tested) functions
rank_lookup = textprep.build_rank_lookup(textprep.tokenize(gutenberg.raw()))

samples = {
    "Austen - Emma (ch. 1-3)": emma_raw[:20_000],
    "sci.med posts": " ".join(d for d, t in zip(news.data, news.target)
                              if news.target_names[t] == "sci.med")[:20_000],
    "rec.sport.baseball posts": " ".join(d for d, t in zip(news.data, news.target)
                                         if news.target_names[t] == "rec.sport.baseball")[:20_000],
}

profiles = pd.DataFrame({
    name: textprep.lexical_profile(textprep.tokenize(text), rank_lookup).set_index("band")["share"]
    for name, text in samples.items()
})
(profiles * 100).round(1)

,Austen - Emma (ch. 1-3),sci.med posts,rec.sport.baseball posts
band,,,
A1,75.8,58.8,61.0
A2,6.5,6.0,4.7
B1,6.6,5.5,4.9
B2,5.2,3.8,5.5
C1,3.4,5.3,4.4
C2,2.5,10.5,8.9
UNKNOWN,0.0,10.0,10.6


**Leitura:** a maior fatia em A1 é esperada em todos (palavras funcionais dominam qualquer texto).
Os posts de fórum tendem a ter **mais `UNKNOWN`** — termos médicos, nomes de jogadores, gírias e erros de digitação que
não aparecem nos clássicos do Gutenberg. Isso mostra um ponto importante de PLN aplicado à educação: **a escolha do corpus
de referência muda a classificação de dificuldade.** Um próximo passo seria usar um corpus contemporâneo de referência.

In [22]:
# Final run of the whole suite, as a CI pipeline would do
final_exit = run_pytest("-q", "--cov=textprep", "--cov-report=term")
assert final_exit == 0, "the full test suite must pass"
print("Test files in the lab project:", sorted(p.name for p in WORKDIR.glob("test_*.py")))

.......................................................................  [100%]
================================ tests coverage ================================
_______________ coverage: platform linux, python 3.11.6-final-0 ________________

Name          Stmts   Miss  Cover
---------------------------------
textprep.py      74      0   100%
---------------------------------
TOTAL            74      0   100%
71 passed in 3.64s

exit code: 0
Test files in the lab project: ['test_basics.py', 'test_corpus.py', 'test_dataframes.py', 'test_exceptions.py', 'test_fixtures.py', 'test_parametrize.py', 'test_scope.py']


### 📝 Resumo do dia

| Conceito | O que aprendi | Exemplo no notebook |
|---|---|---|
| `assert` + descoberta | funções `test_*` em arquivos `test_*.py` | `test_basics.py` |
| Introspecção | o pytest mostra os valores quando um teste falha | demo `0.5 == 0.75` |
| `pytest.raises` | entradas inválidas devem gerar erros claros | `clean_text(NaN)` |
| Fixtures | dados reutilizáveis, encadeáveis, com escopo e *teardown* | *Emma*, *Persuasion* |
| `tmp_path` | pasta temporária para testar I/O | CSV *round-trip* |
| `parametrize` | muitos casos e valores-limite sem repetição | faixas 500/501, 1000/1001 |
| pandas/numpy | `assert_frame_equal`, `assert_allclose`, `approx` | `word_frequencies` |
| `conftest.py` | fixtures compartilhadas de sessão + hooks | Gutenberg, 20 Newsgroups |
| Markers / `-k` | rodar subconjuntos (`slow`) | `-m "not slow"` |
| `pytest-cov` | mapa das linhas não testadas; *gate* no CI | `--cov-fail-under=90` |
| `ipytest` | testes rápidos dentro do notebook | *Alice in Wonderland* |

### ✅ Checklist para meus próximos projetos de PLN

- [ ] Funções de pré-processamento em um módulo `.py` (não só em células de notebook).
- [ ] Um teste por comportamento, com nome descritivo.
- [ ] Casos-limite linguísticos: apóstrofos tipográficos, acentos, contrações, texto vazio, `NaN`.
- [ ] Fixtures de sessão para corpora grandes; marcar testes lentos com `@pytest.mark.slow`.
- [ ] Testar o **contrato** dos DataFrames (colunas e dtypes), não só os valores.
- [ ] Cobertura como guia, com *gate* razoável no CI (GitHub Actions).

### 🔭 Próximos passos
- Integrar `pytest` ao GitHub Actions deste portfólio.
- Explorar `hypothesis` (*property-based testing*) para gerar textos aleatórios automaticamente.
- Testar *pipelines* do scikit-learn (ex.: `TfidfVectorizer` com o meu tokenizador customizado).

**Referências:** documentação oficial do [pytest](https://docs.pytest.org/), [pytest-cov](https://pytest-cov.readthedocs.io/),
[NLTK corpora](https://www.nltk.org/nltk_data/) e [20 Newsgroups no scikit-learn](https://scikit-learn.org/stable/datasets/real_world.html#newsgroups-dataset).